# 02 · BEV & Sensor Fusion：为什么要选择空间表示？

本章合并旧版 `00C`、`02` 和 `06`。目标不是背 BEV 模型名，而是亲手完成：

```text
camera/LiDAR observations → ego-frame BEV grid → occupancy/risk targets
```

这份 grid 会被 `05 Learnable BEV Model` 直接读取训练；它不再是与前面无关的随机 token 分类题。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ad_tutorial import (
    ARTIFACT_DIR,
    BEVConfig,
    build_bev_dataset,
    build_urban_cut_in_scene,
    ensure_artifact_dir,
    load_json_artifact,
    load_numpy_artifact,
    save_json_artifact,
    save_numpy_artifact,
    scene_to_bev,
)

ensure_artifact_dir()
print("project root:", PROJECT_ROOT)
print("artifact directory:", ARTIFACT_DIR)

import numpy as np
import matplotlib.pyplot as plt

scene = build_urban_cut_in_scene(seed=7, timestamp_s=0.0)
config = BEVConfig(resolution=1.0)
encoded = scene_to_bev(scene, config)
print("BEV shape:", encoded["camera"].shape, "feature channels:", encoded["features"].shape)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, image, title in zip(axes, [encoded["camera"], encoded["lidar"], encoded["targets"][1]],
                            ["camera occupancy", "LiDAR occupancy", "cut-in risk target"]):
    ax.imshow(image.T, origin="lower", aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("y cell")
    ax.set_ylabel("x cell")
plt.tight_layout()
plt.show()

## 1. Representation and fusion are different choices

- occupancy preserves “where is space occupied?” but not necessarily class, intent, or topology;
- object boxes preserve instance semantics but may hide free space;
- vector/map representations preserve lanes and topology but depend on map quality;
- agent state preserves identity/velocity and feeds prediction.

Fusion can happen before rasterization, in a shared BEV feature space, or after separate heads. Whatever the architecture, calibration/time quality and modality health must be visible to the model or safety layer.

In [ ]:
from ipywidgets import FloatSlider, interact

def fusion_experiment(resolution=1.0, camera_dropout=0.0, time_lag=0.0):
    local = build_urban_cut_in_scene(seed=7, timestamp_s=0.0, sensor_lag_s=time_lag)
    cfg = BEVConfig(resolution=resolution)
    result = scene_to_bev(local, cfg)
    rng = np.random.default_rng(20)
    camera = result["camera"].copy()
    camera[rng.random(camera.shape) < camera_dropout] = 0.0
    fused = np.maximum(camera, result["lidar"])
    target = result["targets"][0] > 0
    iou = np.logical_and(fused > 0, target).sum() / max(np.logical_or(fused > 0, target).sum(), 1)
    print(f"resolution={resolution:.1f}m, camera dropout={camera_dropout:.2f}, lag={time_lag:.2f}s, fused IoU={iou:.3f}")
    print("failure interpretation: resolution changes quantization; dropout removes evidence; lag shifts evidence")

interact(
    fusion_experiment,
    resolution=FloatSlider(min=0.5, max=2.0, step=0.5, value=1.0),
    camera_dropout=FloatSlider(min=0.0, max=0.9, step=0.1, value=0.0),
    time_lag=FloatSlider(min=0.0, max=0.4, step=0.05, value=0.0),
)

## 2. Real-data checkpoint: construct one nuScenes BEV

The public checkpoint is deliberately small: use the official devkit to read one `LIDAR_TOP` sample from nuScenes mini, transform points into ego coordinates, and pass the resulting `N×3` array through `rasterize_points`. Compare its point count, range and empty-cell pattern with the toy scene. Do not call the toy occupancy grid a benchmark result.

In [ ]:
from ad_tutorial.scene import rasterize_points
import os

def real_data_bev_checkpoint(points_xyz=None, sample_token=None):
    if points_xyz is None:
        root = os.environ.get("NUSCENES_ROOT")
        if not root:
            print("Set NUSCENES_ROOT or pass ego-frame points to run the real-data checkpoint.")
            return None
        try:
            from nuscenes.nuscenes import NuScenes
            from nuscenes.utils.data_classes import LidarPointCloud
            from pyquaternion import Quaternion
        except ImportError as exc:
            print("Install requirements-real-data.txt first:", exc)
            return None
        nusc = NuScenes(version="v1.0-mini", dataroot=root, verbose=False)
        sample = nusc.sample[0] if sample_token is None else nusc.get("sample", sample_token)
        lidar_sd = nusc.get("sample_data", sample["data"]["LIDAR_TOP"])
        lidar_cs = nusc.get("calibrated_sensor", lidar_sd["calibrated_sensor_token"])
        ego_pose = nusc.get("ego_pose", lidar_sd["ego_pose_token"])
        cloud = LidarPointCloud.from_file(str(Path(root) / lidar_sd["filename"]))
        cloud.rotate(Quaternion(lidar_cs["rotation"]).rotation_matrix)
        cloud.translate(np.asarray(lidar_cs["translation"]))
        cloud.rotate(Quaternion(ego_pose["rotation"]).rotation_matrix)
        cloud.translate(np.asarray(ego_pose["translation"]))
        # Move global points back to the ego frame at the sample time.
        cloud.translate(-np.asarray(ego_pose["translation"]))
        cloud.rotate(Quaternion(ego_pose["rotation"]).inverse.rotation_matrix)
        points_xyz = cloud.points[:3].T
    real_grid = rasterize_points(np.asarray(points_xyz), BEVConfig(resolution=1.0))
    print("real-data BEV shape:", real_grid.shape, "occupied cells:", int(real_grid.sum()))
    return real_grid

real_data_bev_checkpoint()

## 3. Produce the training artifact for Chapter 05

Use a coarser `2m` grid for a CPU-friendly but spatially meaningful Transformer. Every sample is a variant of the same cut-in task; labels are occupancy, cut-in risk, and x velocity. The exact array shape and coordinate convention are part of the artifact contract.

In [ ]:
train_config = BEVConfig(resolution=2.0)
dataset = build_bev_dataset(n=72, seed=101, config=train_config)
save_numpy_artifact("02_bev_dataset.npz", features=dataset["features"], targets=dataset["targets"])
save_json_artifact("02_bev_dataset_meta.json", {
    "scenario": dataset["scenario"],
    "config": dataset["config"],
    "features": ["camera_occupancy", "lidar_occupancy", "camera_age"],
    "targets": ["occupancy", "cut_in_risk", "velocity_x"],
    "next": "05_learnable_bev_model.ipynb",
})
print("saved training features:", dataset["features"].shape)
print("saved training targets:", dataset["targets"].shape)

### 完成标准

你应能回答：为什么 BEV 对 planning 友好但不是所有任务的唯一表示？错位和 dropout 如何传播？`02_bev_dataset.npz` 中每个 channel/target 的含义是什么？下一章会将 temporal state 与 tracking 接到同一场景，随后 Chapter 05 会真的训练这些 BEV targets。